<a href="https://colab.research.google.com/github/rennykurian/AI_Job_Management_Agent_2/blob/main/Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on 1 · Quantization: the same model, three sizes (~10–15 min)

**The question:** if we store a model's numbers with fewer bits, how much memory do we save, and what do we lose?

**What we do:**
1. A tiny demo: round some numbers to 8-bit and 4-bit and see the error.
2. Load the **same model** (Qwen2.5-3B-Instruct) three ways: **16-bit, 8-bit, 4-bit**.
3. For each: memory used, speed, a quality score, and the answers to the same 3 questions.

**Lab note:** production models are often 7B–70B. We use a 3B model so all three versions load on Colab's T4 in a few minutes. The method is identical at any size.

**Before anything:** Runtime → Change runtime type → **T4 GPU** → Save. Then **Runtime → Run all**. No keys needed.

## Step 1 · Install (~1 min)
`bitsandbytes` is the library that does 8-bit and 4-bit loading.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
import torch, time, gc, math
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


## Step 2 · Quantization in miniature (instant)
**What's happening:** a model is millions of decimal numbers (weights). Quantizing means storing each one with fewer bits by **rounding it to a small set of levels**. We do that to 1 million random "weights" and measure how far the rounded values drift from the originals.

**What you'll see:** 8-bit keeps numbers very close (tiny error) at half the size; 4-bit is 4× smaller with a bigger, but still small, error.

In [ ]:
w = torch.randn(1_000_000) * 0.02          # pretend weights: small numbers around 0, like a real model

def quantize(x, bits):
    levels = 2 ** (bits - 1) - 1              # 8-bit -> 127 levels each side, 4-bit -> 7
    scale = x.abs().max() / levels            # one scale for the whole block
    q = torch.round(x / scale).clamp(-levels, levels)   # the small integers we actually store
    return q * scale                          # turn back into decimals when computing

print(f"{'format':8s} {'bits/number':>12s} {'size vs 16-bit':>15s} {'average error':>15s}")
for bits in [16, 8, 4]:
    err = 0.0 if bits == 16 else (quantize(w, bits) - w).abs().mean().item()
    print(f"{str(bits) + '-bit':8s} {bits:12d} {bits / 16:14.0%} {err:15.6f}")
print("\nexample weight:", round(w[0].item(), 6), "-> 8-bit:", round(quantize(w, 8)[0].item(), 6), " 4-bit:", round(quantize(w, 4)[0].item(), 6))

format    bits/number  size vs 16-bit   average error
16-bit             16           100%        0.000000
8-bit               8            50%        0.000198
4-bit               4            25%        0.003601

example weight: 0.002256 -> 8-bit: 0.002381  4-bit: 0.0


## Step 3 · Load the real model three ways (~6–10 min, mostly downloading)
**What's happening:** Qwen2.5-3B-Instruct has about **3 billion** numbers. We load it in 16-bit, then 8-bit, then 4-bit (NF4, the format QLoRA uses). For each we measure:
- **memory**: how much GPU memory the weights take
- **speed**: word pieces (tokens) generated per second
- **quality**: *perplexity* on a short paragraph. Lower = the model finds normal English less surprising. Small changes = quality kept.
- the **answers** to 3 fixed questions (no randomness)

**What you'll see:** memory dropping roughly 6 GB → 3.5 GB → 2 GB, perplexity nearly unchanged, and very similar answers. Speed may *not* improve: on a T4 the 8-bit and 4-bit versions are often slower, because numbers are unpacked on the fly.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

QUESTIONS = ["Explain what a GPU is in two sentences.",
             "What is 17 multiplied by 23? Answer with just the number.",
             "Write a two-line poem about rain."]
PARAGRAPH = ("The library opened early on Saturday mornings, and the first visitors were usually students "
             "preparing for exams. They chose quiet tables near the windows, spread out their notes, and worked "
             "until lunch, when the reading room slowly filled with families looking for picture books.")

CONFIGS = {
    "16-bit": dict(dtype=torch.float16),
    "8-bit":  dict(quantization_config=BitsAndBytesConfig(load_in_8bit=True)),
    "4-bit":  dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                          bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)),
}

def generate(model, q, n=60):
    prompt = tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    x = tok(prompt, return_tensors="pt").to("cuda")
    torch.cuda.synchronize(); t = time.time()
    with torch.no_grad():
        out = model.generate(**x, max_new_tokens=n, do_sample=False, pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize()
    new = out[0][x["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), len(new) / (time.time() - t)

def perplexity(model):
    x = tok(PARAGRAPH, return_tensors="pt").to("cuda")
    with torch.no_grad():
        return math.exp(model(**x, labels=x["input_ids"]).loss.item())

results = {}
for name, kwargs in CONFIGS.items():
    print(f"Loading {name} ...")
    model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="cuda", **kwargs)
    model.eval()
    answers, speeds = [], []
    for q in QUESTIONS:
        a, s = generate(model, q); answers.append(a); speeds.append(s)
    results[name] = {"memory_gb": model.get_memory_footprint() / 1e9, "tokens_per_s": sum(speeds) / len(speeds),
                     "perplexity": perplexity(model), "answers": answers}
    print(f"  memory {results[name]['memory_gb']:.1f} GB | speed {results[name]['tokens_per_s']:.1f} tok/s | perplexity {results[name]['perplexity']:.2f}")
    del model; gc.collect(); torch.cuda.empty_cache()

KeyboardInterrupt: 

## Step 4 · Compare: before (16-bit) vs after (8-bit, 4-bit)
**What you'll see:** one summary table, then each question with the three answers under each other.

In [ ]:
base = results["16-bit"]
print(f"{'format':8s} {'memory':>9s} {'vs 16-bit':>10s} {'speed':>11s} {'perplexity':>11s}")
for name, r in results.items():
    print(f"{name:8s} {r['memory_gb']:7.1f}GB {r['memory_gb'] / base['memory_gb']:9.0%} {r['tokens_per_s']:8.1f}t/s {r['perplexity']:11.2f}")

for i, q in enumerate(QUESTIONS):
    print("\n" + "=" * 80 + "\nQ:", q)
    for name, r in results.items():
        print(f"\n[{name}] {r['answers'][i]}")

format      memory  vs 16-bit       speed  perplexity
16-bit       6.2GB      100%     15.1t/s       18.97
8-bit        3.4GB       55%      1.3t/s       19.15
4-bit        2.0GB       33%      6.6t/s       22.24

Q: Explain what a GPU is in two sentences.

[16-bit] A GPU, or Graphics Processing Unit, is a specialized electronic circuit designed to rapidly manipulate and alter memory to accelerate the creation of images, videos, and animations in a computer. It is optimized for parallel processing, making it particularly well-suited for tasks that involve complex calculations and data manipulation, such as

[8-bit] A GPU, or Graphics Processing Unit, is a specialized electronic circuit designed to rapidly manipulate and alter memory to accelerate the creation of images, videos, and animations in a computer. GPUs are often used for complex calculations and processing tasks that require parallel computing capabilities.

[4-bit] A GPU, or Graphics Processing Unit, is a specialized electro

## What this shows
- **Memory** drops a lot: 4-bit needs roughly a third of 16-bit. That's how a 7B model (~14 GB in 16-bit) runs in ~4 GB, and a 70B model fits on one 80 GB GPU instead of two.
- **Quality** barely moves at 8-bit and 4-bit: perplexity and answers stay close. Below 4 bits, quality drops fast.
- **Speed** isn't automatically better: these bitsandbytes formats save memory, not time. Formats like GGUF (Ollama, llama.cpp), GPTQ and AWQ are built for fast quantized inference.
- **Next:** 4-bit loading is half of QLoRA. The other half is LoRA.